In [3]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. Đọc dữ liệu
print("Đang đọc dữ liệu...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

target_col = "is_fraud"
id_col = "id"
feature_cols = [c for c in train.columns if c not in [target_col, id_col]]

X = train[feature_cols]
y = train[target_col].astype(int)

# 2. Phân loại feature số và categorical / boolean
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

# 3. Tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

# 4. Pipeline với RandomForestClassifier
# Chú ý: Dữ liệu fraud thường rất mất cân bằng (imbalanced), dùng class_weight='balanced' hoặc 'balanced_subsample'
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=16,
                min_samples_split=10,
                min_samples_leaf=5,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42,
                verbose=1,
            ),
        ),
    ]
)

# 5. Chia tập Train - Validation (Stratified để giữ tỉ lệ fraud)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Huấn luyện trên tập Train
print("Bắt đầu huấn luyện Random Forest...")
rf_model.fit(X_train, y_train)

# 7. Đánh giá trên tập Validation
print("\n--- ĐÁNH GIÁ TRÊN TẬP VALIDATION ---")
y_val_prob = rf_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_prob >= 0.5).astype(int)

# 4 chỉ số theo yêu cầu:
roc_auc = roc_auc_score(y_val, y_val_prob)
auc_pr = average_precision_score(y_val, y_val_prob)
precision = precision_score(y_val, y_val_pred, zero_division=0)
recall = recall_score(y_val, y_val_pred, zero_division=0)

print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Precision : {precision:.4f} (với threshold = 0.5)")
print(f"Recall    : {recall:.4f} (với threshold = 0.5)")

print("\nChi tiết Classification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

# 8. Dự đoán trên test.csv và xuất file submission
print("Đang dự đoán trên tập test.csv...")
test_prob = rf_model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({id_col: test[id_col], target_col: test_prob})
submission.to_csv("submission_rf.csv", index=False)
print("Đã lưu kết quả dự đoán vào submission_rf.csv thành công!")

Đang đọc dữ liệu...
Bắt đầu huấn luyện Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   11.4s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   32.5s finished



--- ĐÁNH GIÁ TRÊN TẬP VALIDATION ---


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.3s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    1.1s finished


ROC-AUC   : 0.9361
AUC-PR    : 0.4322
Precision : 0.1457 (với threshold = 0.5)
Recall    : 0.7686 (với threshold = 0.5)

Chi tiết Classification Report:
              precision    recall  f1-score   support

           0     0.9958    0.9234    0.9582    137654
           1     0.1457    0.7686    0.2449      2338

    accuracy                         0.9208    139992
   macro avg     0.5707    0.8460    0.6016    139992
weighted avg     0.9816    0.9208    0.9463    139992

Đang dự đoán trên tập test.csv...


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    2.3s finished


Đã lưu kết quả dự đoán vào submission_rf.csv thành công!
